# Signal Tests (work/02_signal_tests)

This notebook loads the cached aggregate (work/outputs/labelled_agg.parquet or eda_starter.parquet) and runs simple signal association tests: grouped means, effect sizes, and per-client stability checks.


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
Path('work/outputs').mkdir(parents=True, exist_ok=True)
# Load cached file (labelled file created from warehouse). If not present, load starter cache.
import os
if os.path.exists('work/outputs/labelled_agg.parquet'):
    df = pd.read_parquet('work/outputs/labelled_agg.parquet')
elif os.path.exists('work/outputs/eda_starter.parquet'):
    df = pd.read_parquet('work/outputs/eda_starter.parquet')
else:
    raise FileNotFoundError('No cached input found. Run the starter EDA or the warehouse aggregation step.')
print('Loaded for signal tests, rows:', len(df))


In [ ]:
# Example signal: avg_position buckets vs CTR (if available)
if 'avg_position' in df.columns and 'ctr' in df.columns:
    df['pos_bucket'] = pd.cut(df['avg_position'], bins=[0,3,5,10,20,50,9999], labels=['1-3','4-5','6-10','11-20','21-50','50+'])
    grouped = df.groupby('pos_bucket')['ctr'].agg(['count','mean','std']).reset_index()
    display(grouped)

# Cohen's d function for effect size
def cohens_d(a, b):
    na, nb = len(a), len(b)
    sa, sb = a.std(ddof=1), b.std(ddof=1)
    pooled = np.sqrt(((na-1)*sa*sa + (nb-1)*sb*sb) / (na+nb-2))
    return (a.mean() - b.mean()) / pooled if pooled>0 else 0

# Example: top vs bottom decile of position (requires non-null avg_position)
if 'avg_position' in df.columns and 'ctr' in df.columns:
    valid = df.dropna(subset=['avg_position','ctr'])
    q_low = valid['avg_position'].quantile(0.1)
    q_high = valid['avg_position'].quantile(0.9)
    low_group = valid[valid['avg_position'] <= q_low]['ctr']
    high_group = valid[valid['avg_position'] >= q_high]['ctr']
    print('Cohen d (low vs high position) on CTR:', cohens_d(low_group, high_group))
